# Regressão Logística
---

## A) Importando as Bibliotecas e Pacotes a serem usados
---

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from imblearn.over_sampling import SMOTE

## B) Determinando os Dataframes à partir das bases de dados pré-processados
---

### 1. Determinando df_treino
---

In [2]:
df_treino = pd.read_parquet('../../dataset/treino_pre_processado.parquet')
df_treino.info()

<class 'pandas.DataFrame'>
Index: 96822 entries, 0 to 175337
Data columns (total 36 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   duracao_conexao_seg                 96822 non-null  float64
 1   protocolo_rede                      96822 non-null  float64
 2   servico_conexao                     96822 non-null  float64
 3   estado_conexao                      96822 non-null  float64
 4   num_pacotes_origem                  96822 non-null  float64
 5   num_pacotes_destino                 96822 non-null  float64
 6   total_bytes_origem                  96822 non-null  float64
 7   total_bytes_destino                 96822 non-null  float64
 8   taxa_pacote                         96822 non-null  float64
 9   carga_origem                        96822 non-null  float64
 10  carga_destino                       96822 non-null  float64
 11  pacotes_perdidos_origem             96822 non-null  floa

### 2. Deteminando df_teste
---

In [3]:
df_teste = pd.read_parquet('../../dataset/teste_pre_processado.parquet')
df_teste.info()

<class 'pandas.DataFrame'>
Index: 49971 entries, 0 to 82328
Data columns (total 36 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   duracao_conexao_seg                 49971 non-null  float64
 1   protocolo_rede                      49971 non-null  float64
 2   servico_conexao                     49971 non-null  float64
 3   estado_conexao                      49971 non-null  float64
 4   num_pacotes_origem                  49971 non-null  float64
 5   num_pacotes_destino                 49971 non-null  float64
 6   total_bytes_origem                  49971 non-null  float64
 7   total_bytes_destino                 49971 non-null  float64
 8   taxa_pacote                         49971 non-null  float64
 9   carga_origem                        49971 non-null  float64
 10  carga_destino                       49971 non-null  float64
 11  pacotes_perdidos_origem             49971 non-null  float

## C) Separando os valores em teste e treino
---

### 1. Atributos do df_treino
---

In [4]:
# Features
X_treino = df_treino.drop(columns='label')
# X_treino = df_treino.drop(columns='categoria_ataque')

# Target
y_treino = df_treino['label']
# y_treino = df_treino['categoria_ataque']

### 2. Atributos do df_teste
---

In [5]:
# Features
X_teste = df_teste.drop(columns='label')
# X_teste = df_teste.drop(columns='categoria_ataque')

# Target
y_teste = df_teste['label']
# y_teste = df_teste['categoria_ataque']

### D) Visualização dos dados
---

In [6]:
# fig, ax = plt.subplots()
    
# ax.scatter(df_treino['pacotes_perdidos_origem'], df_treino['categoria_ataque'])
    
# ax.set(yticks=(0,1))
    
# plt.show()

In [7]:
# for x in X_treino.columns:

#     print(f'Atributo {x.upper()}:\n\n')
    
#     fig, ax = plt.subplots()
    
#     plt.scatter(df_treino[x], y_treino)
    
#     ax.set(yticks=(0,10))

#     print(f'='*30)
#     plt.show()

## E) Desenvolvimento do Modelo
---

### 1. Aplicação do SMOTE para desbalanceamento de classes nas variáveis de treino
---

#### 1.1. Instanciamento da classe SMOTE
---

In [8]:
sm = SMOTE(random_state=42)

#### 1.2. Reamostragem para balanceamento do conjuto de dados
---

In [9]:
X_treino_balanceado, y_treino_balanceado = sm.fit_resample(X_treino, y_treino)

#### 1.3. Conferência do balanceamento do Target categoria_ataque
---

In [10]:
print(f'Target CATEGORIA_ATAQUE:\n')
print(f'- Antes da aplicação do SMOTE:\n \n{y_treino.value_counts()}\n')
print(f'- Depois da aplicação do SMOTE:\n \n{y_treino_balanceado.value_counts()}\n')

Target CATEGORIA_ATAQUE:

- Antes da aplicação do SMOTE:
 
label
0.0    48894
1.0    47928
Name: count, dtype: int64

- Depois da aplicação do SMOTE:
 
label
0.0    48894
1.0    48894
Name: count, dtype: int64



### 2. Aplicação de PolynomialFeatures
---

#### 2.1. Criação da instância da função PolynomialFeatures
---

In [11]:
poly = PolynomialFeatures(degree=2)

#### 2.2. Aprendizagem e transformação com as features do Dataset de Treino
---

In [12]:
X_treino_poly = poly.fit_transform(X_treino_balanceado)
X_treino_poly

array([[ 1.00000000e+00, -5.42402662e-03,  1.13000000e+02, ...,
         0.00000000e+00,  0.00000000e+00,  3.60000000e+01],
       [ 1.00000000e+00,  3.44877546e-03,  1.13000000e+02, ...,
         0.00000000e+00,  0.00000000e+00,  3.60000000e+01],
       [ 1.00000000e+00,  1.97902947e-02,  1.13000000e+02, ...,
         0.00000000e+00,  0.00000000e+00,  3.60000000e+01],
       ...,
       [ 1.00000000e+00,  1.83888456e-02,  1.13000000e+02, ...,
         0.00000000e+00,  0.00000000e+00,  4.00000000e+00],
       [ 1.00000000e+00,  3.64165503e-03,  1.13000000e+02, ...,
         0.00000000e+00,  0.00000000e+00,  4.90000000e+01],
       [ 1.00000000e+00,  1.84677648e-03,  1.13000000e+02, ...,
         0.00000000e+00,  0.00000000e+00,  9.00000000e+00]],
      shape=(97788, 666))

#### 2.3. Transformação com as features do Dataset de Teste
---

O teste só é transformado para que o modelo aprenda apenas os dados de treino e evitar, assim, o `overfitting` que diminuiria o desempenho dele com dados reais.

In [13]:
X_teste_poly = poly.transform(X_teste)
X_teste_poly

array([[ 1.00000000e+00, -8.01948672e-03,  1.17000000e+02, ...,
         0.00000000e+00,  0.00000000e+00,  3.60000000e+01],
       [ 1.00000000e+00, -8.01953712e-03,  1.17000000e+02, ...,
         0.00000000e+00,  0.00000000e+00,  3.60000000e+01],
       [ 1.00000000e+00, -8.01958752e-03,  1.17000000e+02, ...,
         0.00000000e+00,  0.00000000e+00,  3.60000000e+01],
       ...,
       [ 1.00000000e+00,  5.62075292e-03,  1.11000000e+02, ...,
         0.00000000e+00,  0.00000000e+00,  3.60000000e+01],
       [ 1.00000000e+00,  7.87945255e-02,  1.11000000e+02, ...,
         0.00000000e+00,  0.00000000e+00,  3.60000000e+01],
       [ 1.00000000e+00,  1.05631921e-02,  1.11000000e+02, ...,
         0.00000000e+00,  0.00000000e+00,  3.60000000e+01]],
      shape=(49971, 666))

#### 2.4. Normalização de X_treino_poly e X_teste_poly com Standard Scaler
---

##### 2.4.1. Instanciamento da classe
---

In [14]:
escalonador = StandardScaler()

##### 2.4.2. Normalização de X_treino_poly
---

In [15]:
X_treino_poly_norm = escalonador.fit_transform(X_treino_poly) # deve ser aprendido e transformado

##### 2.4.3. Normalização de X_teste_poly
---

In [16]:
X_teste_poly_norm = escalonador.transform(X_teste_poly) # deve ser apenas transformado

### 3. Treino do modelo
---

#### 3.1. Instanciamento da classe LogisticRegression
---

In [17]:
modelo = LogisticRegression(random_state=42, max_iter=3000, solver='saga', penalty='elasticnet', l1_ratio=0.5, C=5.0)

#### 3.2. Treino do modelo de Regressão Logística as variáveis pré-processadas com SMOTE e PolynomialFeatures
---

In [ ]:
modelo.fit(X_treino_poly_norm, y_treino_balanceado)

### 4. Predição com base de testes
---

In [ ]:
y_predicao = modelo.predict(X_teste)

### 5. Pontuação, Relatório de Classificação e Matriz de Confusão
---

#### 5.1. Pontuação do Modelo (Acurácia Geral)
---

In [ ]:
modelo.score(X_teste, y_teste)

#### 5.2. Relatório de Classificação
---

In [ ]:
print(classification_report(y_teste, y_predicao))

#### 5.3. Matriz de Confusão
---

In [ ]:
matriz_confusao = confusion_matrix(y_teste, y_predicao)

In [ ]:
disp = ConfusionMatrixDisplay(confusion_matrix=matriz_confusao, display_labels=modelo.classes_)
disp.plot(cmap=plt.cm.Blues)
plt.show()

In [ ]:
(tn, fp), (fn, tp) = matriz_confusao
total = (tn+fn+tp+fp)

In [ ]:
def calcular_percentual(var, total):
    percentual = (((var)/(total))*100)
    return f'{percentual:.2f}'

In [ ]:
print(f'O total de predições de teste foi {total}.')
print(f'\n - Verdadeiro Negativo (TN): Predição correta sobre o diagnóstico negativo. Há {tn} casos que o modelo apontou como resultado negativo, corretamente.')
print(f'\n - Falso Negativo (FN): Predição incorreta sobre o diagnóstico negativo. Há {fn} pacientes em que o modelo apontou como resultado negativo, erroneamente. Maior risco pois, possivelmente, o usuário não buscará ajuda para o caso de ataque cibernético, por acreditar não ser necessário.')
print(f'\n - Verdadeiro Positivo (TP): Predição correta sobre o diagnóstico positivo. Há {tn} pacientes em que o modelo apontou como resultado positivo, corretamente.')
print(f'\n - Falso Positivo (FP): Predição incorreta sobre o diagnóstico positivo. Há {fp} pacientes em que o modelo apontou como resultado positivo, erroneamente.')
print(f'\n\nResultados:')
print(f'\n - A quantidade de vezes que o modelo acertou é expressa por TN + TP e é igual a {(tn + tp)}, de um total de {total}. Isto corresponde a, aproximadamente, {calcular_percentual((tn+tp), total)}% do total de predições.')
print(f'\n - A quantidade de vezes que o modelo errou é expressa por FN + FP e é igual a {(fn + fp)}, de um total de {total}. Isto corresponde a, aproximadamente, {calcular_percentual((fn+fp), total)}% do total de predições.\n\n')

In [ ]:
print(f'Total TN: {tn}, percentual, sob o total de predições: {(calcular_percentual(tn, total))}%')
print(f'Total TP: {tp}, percentual, sob o total de predições: {calcular_percentual(tp, total)}%')
print(f'Total FN: {fn}, percentual, sob o total de predições: {calcular_percentual(fn, total)}%')
print(f'Total FP: {fp}, percentual, sob o total de predições: {calcular_percentual(fp, total)}%')